# درس ۱۸: ایمن‌سازی عامل‌های هوش مصنوعی با رسیدهای رمزنگاری‌شده

## دفترچه تمرینی عملی

این دفترچه تمرینی چهار تمرین را طی می‌کند:

۱. **اولین رسید خود را امضا کنید** برای یک تماس ابزار عامل و آن را تأیید کنید.
۲. **با رسید دستکاری کنید** و مشاهده کنید که تأیید شکست می‌خورد.
۳. **زنجیره‌ای از سه رسید بسازید** و صحت زنجیره را تأیید کنید.
۴. **تماس با ابزار Microsoft Agent Framework را wrap کنید** به گونه‌ای که هر اقدام یک رسید صادر کند.

تمامی عناصر رمزنگاری از کتابخانه‌های معتبر وارد شده‌اند (`pynacl` برای Ed25519، `jcs` برای JSON کاننیکال RFC 8785، `hashlib` از کتابخانه استاندارد پایتون برای SHA-256). منطق رسید خودِ پایتون ساده است که می‌توانید آن را بخوانید و تغییر دهید.

سلول‌ها را به ترتیب اجرا کنید. هر بخش کوتاه و خودکفا است.


## راه‌اندازی

دو وابستگی را نصب کنید. هر دو دارای مجوزهای آزاد (Apache-2.0 / MIT) هستند.


In [1]:
%pip install -q pynacl jcs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import CryptoError
from jcs import canonicalize

## ابزارهای کمکی

این دو ابزار کمکی کدگذاری base64url (بدون پدینگ) و هش SHA-256 اشیاء دلخواه را انجام می‌دهند. آن‌ها بقیه دفترچه یادداشت را متمرکز بر منطق رسید نگه می‌دارند.


In [3]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """
    Decode unpadded base64url strictly, raising ValueError for anything else.

    Python's decoder skips characters outside the alphabet and also accepts
    padding, the standard +/ alphabet and non-zero spare bits, so the same key
    or signature could be written several ways. Re-encoding and comparing
    accepts only the one form that b64url_nopad produces.
    """
    padding = "=" * ((4 - len(s) % 4) % 4)
    data = base64.urlsafe_b64decode(s + padding)
    if b64url_nopad(data) != s:
        raise ValueError("not canonical unpadded base64url")
    return data

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## بخش 1: اولین رسید خود را امضا کنید

تصور کنید نماینده ما برای **Contoso Travel** به تازگی پروازهایی از سیدنی به لس‌آنجلس را برای مشتری جستجو کرده است. ما می‌خواهیم بیانیه دروازه درباره این تماس ابزار را ثبت کنیم تا در آینده یک حسابرس بتواند آن را با یک کلید صادرکننده پین‌شده جداگانه تأیید کند.

### مرحله 1.1: ایجاد یک کلید امضا

در محیط تولید، کلید امضای نماینده معمولاً در یک ماژول امنیت سخت‌افزاری (HSM)، Azure Key Vault، یا فروشگاه محافظت‌شده مشابهی نگهداری می‌شود. برای این درس ما یک کلید تازه را در حافظه تولید می‌کنیم.


In [4]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
# Trust configuration is out of band. Never trust a receipt only because it embeds a key.
TRUSTED_PUBLIC_KEYS = {public_key_b64}
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

Public key (Ed25519, 32 bytes): NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM


### گام 1.2: ساخت بار داده رسید

بار داده شامل همه چیزهایی است که می‌خواهیم رسید به آن تصدیق کند: چه کسی عمل کرده است، چه ابزاری، با چه آرگومان‌هایی، چه نتیجه‌ای بازگشته است، تحت چه سیاستی، و چه زمانی. ما به جای وارد کردن آرگومان‌ها و نتیجه به صورت خطی، هش آن‌ها را می‌گیریم تا رسید اطلاعات حساس را افشا نکند.


In [5]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null
}


### گام ۱.۳: امضا و مونتاژ رسید

سه گام:

۱. مقدار داده (payload) را با استفاده از JCS کانونیکال کنید تا دو پیاده‌سازی که همان رسید منطقی را تولید می‌کنند، بایت‌هایی کاملاً یکسان تولید کنند.
۲. بایت‌های کانونیکال شده را مستقیماً با کلید خصوصی Ed25519 امضا کنید. PureEdDSA به صورت داخلی پیام را هش می‌کند، بنابراین هش کردن اضافی قبل از امضا، پروتکل را تغییر می‌دهد.

سپس امضا به مقدار داده اصلی پیوست می‌شود تا رسید نهایی تولید شود.


In [6]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    signature_bytes = signing_key.sign(canonical).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null,
  "signature": {
    "alg": "EdDSA",
    "sig": "NnH0T6lpjVThg1RS29rEu3OG1FptWbQgX18Yt1JAPjNnTyje2Luozt5_0gcdxddQXoS6eH6Vlg5MTcKLrvWnBQ",
    "public_key": "NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM"
  }
}


### مرحله 1.4: تأیید رسید

تأیید فرآیند را معکوس می‌کند. ما امضا را جدا می‌کنیم، بایت‌های اصلی را دوباره محاسبه می‌کنیم و امضا را با کلید عمومی صادرکننده بررسی می‌کنیم.

تأیید به‌صورت آفلاین انجام می‌شود: نیازی به تماس با سرویس یا دسترسی به شبکه نیست. اما هنوز یک چیز به جز رسید نیاز دارد: کلید عمومی صادرکننده، که از طریق کانالی که بازرس قبلاً اعتماد دارد به دست می‌آید، مانند مجموعه JWK که صادرکننده منتشر می‌کند یا مقداری که قبلاً پین شده است. فیلد `public_key` در داخل رسید نمی‌تواند این نقش را ایفا کند، زیرا هر کسی که رسید را می‌نویسد، آن فیلد را نیز می‌نویسد. بخش ۲ تقلبی که این روند را جلوگیری می‌کند نشان می‌دهد.


In [7]:
def verify_receipt(receipt: dict, trusted_public_key: str) -> bool:
    """
    Verify a receipt's Ed25519 signature against a key you already trust.

    trusted_public_key is the issuer's key, obtained from somewhere other than
    this receipt: a JWK Set, a pinned config value, a key registry. Reading the
    key out of receipt["signature"]["public_key"] instead would only prove the
    receipt is consistent with itself. Section 2 shows exactly how that fails.

    It fails closed: anything malformed returns False instead of raising, so
    one bad receipt cannot stop a batch of them from being checked.
    """
    if not isinstance(receipt, dict) or not isinstance(trusted_public_key, str):
        return False
    sig_obj = receipt.get("signature")
    if not isinstance(sig_obj, dict) or sig_obj.get("alg") != "EdDSA":
        return False
    sig_b64 = sig_obj.get("sig")
    if not isinstance(sig_b64, str):
        return False

    try:
        # Reconstruct the payload that was actually signed (everything except signature).
        payload = {k: v for k, v in receipt.items() if k != "signature"}
        canonical = canonicalize(payload)
        verify_key = signing.VerifyKey(b64url_decode(trusted_public_key))
        verify_key.verify(canonical, b64url_decode(sig_b64))
        return True
    except CryptoError:
        return False
    except Exception:
        # An undecodable key or signature, or a payload that cannot be
        # canonicalized: fail closed rather than raise.
        return False

# public_key_b64 came from Section 1, alongside the signing key. In a real
# deployment it would come from the issuer's JWK Set, not from this notebook.
is_valid = verify_receipt(receipt, public_key_b64)
print(f"Receipt is valid: {is_valid}")

# Regression control for the signature scope in draft revision 02. A receipt
# signed over SHA-256(JCS(payload)) is a signature over different bytes and
# must not verify as a direct-JCS Ed25519 receipt.
prehashed_signature = signing_key.sign(hashlib.sha256(canonicalize(payload)).digest()).signature
prehashed_receipt = {
    **receipt,
    "signature": {**receipt["signature"], "sig": b64url_nopad(prehashed_signature)},
}
print(f"Pre-hashed receipt valid: {verify_receipt(prehashed_receipt, public_key_b64)}")

# Malformed input is rejected, not raised.
for malformed in ({"signature": "x"}, {"signature": {"alg": "EdDSA"}}, "not a receipt"):
    assert verify_receipt(malformed, public_key_b64) is False
print("Malformed receipts rejected: True")

Receipt is valid: True
Pre-hashed receipt valid: False
Malformed receipts rejected: True


باید `Receipt is valid: True`، `Pre-hashed receipt valid: False` و `Malformed receipts rejected: True` را ببینید. حالت مثبت ثابت می‌کند مسیر امضای مستقیم-JCS کار می‌کند؛ کنترل منفی باعث می‌شود قانون محدوده امضا قابل اجرا شود به جای اینکه به صورت توضیح باقی بماند؛ خط آخر نشان می‌دهد ورودی نامنظم باعث بازگشت `False` می‌شود به جای اینکه یک استثنا ایجاد کند.


## بخش ۲: دستکاری رسید

هدف اصلی رسیدها این است که دستکاری آنها قابل تشخیص باشد. بیایید این موضوع را اثبات کنیم.

ما دقیقاً یک کاراکتر از رسید را تغییر خواهیم داد و شاهد شکست فرآیند تأیید خواهیم بود.


In [8]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered, public_key_b64)}")

Original policy_id:  contoso-travel-policy-v3
Tampered policy_id:  contoso-travel-policy-PERMISSIVE

Tampered receipt valid? False


### چه اتفاقی افتاد؟

وقتی `policy_id` را تغییر دادیم، بایت‌هایcanonical تغییر کردند. امضا (که روی بایت‌های canonical اصلی بود) دیگر تطابق ندارد. اعتبارسنجی به درستی `False` را برمی‌گرداند.

هیچ راهی وجود ندارد که یک فیلد امضا شده، یعنی هر چیزی خارج از شیء `signature`، را تغییر داد و همچنان رسید در برابر کلید عمومی صادرکننده تائید شود، مگر اینکه مهاجم کلید خصوصی صادرکننده را داشته باشد. خود شیء `signature` امضا نشده است: `sig` امضا است، و `verify_receipt` از `public_key` چشم‌پوشی می‌کند، پس فقط ویرایش آن فیلد هیچ تغییری ایجاد نمی‌کند. تا زمانی که کلید خصوصی در یک گنجه کلیدی باشد و اعتباردهنده‌ها از کلید منتشر شده مستقل صادرکننده استفاده کنند، تغییر در یک فیلد امضا شده غیرقابل مخفی‌کردن است. این شرط اهمیت دارد: سلول بعدی یک رسید تغییر یافته را نشان می‌دهد که کاملاً با کلید متفاوتی تائید می‌شود.

خودتان امتحان کنید: `tool_name` یا `agent_id` یا `timestamp` را در سلول بالا تغییر دهید و دوباره اجرا کنید. هر تغییری در یک فیلد امضا شده، رسید نامعتبر تولید می‌کند.


### تغییرناپذیری که امضا کشف نمی‌کند

مهاجم بالا دوباره امضا نکرد. یکی با دقت‌تر این کار را می‌کند: رسید را تغییر می‌دهد،
آن را با کلیدی که خودش ساخته امضا می‌کند، و آن کلید را در
فیلد `public_key` می‌گذارد. نتیجه رسیدی است که امضایش کاملاً واقعی است
برای محتوایش.

این را اجرا کنید و ببینید کدام یک از دو بررسی آن را کشف می‌کند.


In [9]:
# The attacker generates their own keypair. Nothing stops them.
attacker_key = signing.SigningKey.generate()

forged_payload = {k: v for k, v in receipt.items() if k != "signature"}
forged_payload["policy_id"] = "contoso-travel-policy-PERMISSIVE"

# Re-signed, with the attacker's key embedded where the issuer's used to be.
forged = sign_receipt(forged_payload, attacker_key, attacker_key.verify_key)

print(f"Issuer's public key : {public_key_b64}")
print(f"Key inside the forgery: {forged['signature']['public_key']}")
print()
print(f"Checked against the key inside it: "
      f"{verify_receipt(forged, forged['signature']['public_key'])}")
print(f"Checked against the issuer's key : "
      f"{verify_receipt(forged, public_key_b64)}")

# Assert the results, so the cell fails if the forgery ever passes as the issuer's.
assert verify_receipt(receipt, public_key_b64) is True                    # genuine, issuer's key
assert verify_receipt(forged, forged["signature"]["public_key"]) is True  # forgery, its own key
assert verify_receipt(forged, public_key_b64) is False                    # forgery, issuer's key

Issuer's public key : NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM
Key inside the forgery: nIXlWGm0syJQysGkn-nRlzFS59HlPCTqeQr32TJoHM4

Checked against the key inside it: True
Checked against the issuer's key : False


شما باید `True` سپس `False` را ببینید. سه خط `assert` باعث می‌شود که این یک
بررسی باشد: اگر جعل هرگز با کلید صادرکننده اعتبارسنجی شود، سلول خطا می‌دهد.


جعل به صورت داخلی سازگار است. امضای آن واقعاً با
محتوا و کلیدی که حمل می‌کند مطابق است، بنابراین یک تأییدکننده که کلید را از
رسید می‌خواند آن را معتبر گزارش می‌دهد. هیچ چیز در رسید نادرست به نظر نمی‌رسد، زیرا
هیچ چیز در آن ناسازگار نیست.

تنها چیزی که آن را می‌گیرد، بررسی در برابر کلیدی است که مهاجم هیچ
راهی برای تأثیرگذاری روی آن نداشته است. به همین دلیل است که `verify_receipt` کلید را به عنوان آرگومان می‌گیرد: یک
تابع با امضای `verify_receipt(receipt)` به طور ایمن نمی‌تواند نوشته شود،
زیرا تنها کلیدی که می‌تواند به آن دسترسی داشته باشد، کلیدی است که مهاجم ارائه داده است.

همان رسید به عنوان `sample_receipts/04_resigned_receipt.json` ذخیره می‌شود اگر بخواهید
آن را خارج از دفترچه بررسی کنید.


## بخش ۳: رسیدها را به هم زنجیر کنید

یک رسید تنها یک عمل را محافظت می‌کند. بیشتر عامل‌ها چندین عمل انجام می‌دهند. برای اینکه کل دنباله آشکارا دست‌خوردنی نباشد، هر رسید را با شامل کردن هش رسید قبلی در بار داده رسید جدید، به رسید قبلی متصل می‌کنیم.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

اگر کسی رسیدی را در دنباله ارائه شده حذف یا جا‌به‌جا کند، پیوند بعدی شکسته می‌شود. تشخیص کوتاه شدن یا بازنویسی پسوند تحت کنترل امضاکننده نیز به داشتن یک سر جاری ثابت، نقطهٔ بررسی خارجی یا تعهد جمعیتی معتبر نیاز دارد.


In [10]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [11]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

Receipt 0: tool=lookup_flights, prev=None
Receipt 1: tool=hold_seat, prev=sha256:af8b9159b25728881ce6a7166429f4451b9fa869a337453d52228f12131192e7
Receipt 2: tool=confirm_booking, prev=sha256:b1634969232be232f53283251cc9307b443a296909d9e74c2b0101b7f4dc679e


In [12]:
def verify_chain(chain: list, trusted_public_key: str) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt must carry previous_receipt_hash: null for the genesis, and
         the previous receipt's hash for every later receipt.
      3. Each sequence number must be an integer equal to the receipt's zero-based
         position in the chain.
    Returns a list of per-receipt result dicts. A malformed entry fails its own
    checks; it does not stop the rest of the chain from being checked.
    """
    results = []
    for i, receipt in enumerate(chain):
        is_receipt = isinstance(receipt, dict)
        sig_ok = verify_receipt(receipt, trusted_public_key)

        # The link field must be present: .get() alone would read a missing field as None.
        has_link = is_receipt and "previous_receipt_hash" in receipt
        link = receipt.get("previous_receipt_hash") if is_receipt else None
        if i == 0:
            chain_ok = has_link and link is None
        else:
            try:
                chain_ok = has_link and link == receipt_hash(chain[i - 1])
            except Exception:
                chain_ok = False  # the previous entry could not be hashed

        # type() rather than isinstance(): bool is a subclass of int, and True == 1.
        sequence = receipt.get("sequence") if is_receipt else None
        seq_ok = type(sequence) is int and sequence == i
        shape_ok = is_receipt and isinstance(receipt.get("tool_name"), str)

        results.append({
            "index": i,
            "tool": str(receipt.get("tool_name", "?")) if is_receipt else "(not a receipt)",
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok and shape_ok,
        })
    return results

for r in verify_chain(chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): VALID
Receipt 2 (   confirm_booking): VALID


حالا زنجیره را با دستکاری رسید وسط بشکنید و دوباره تأیید کنید. رسید دستکاری شده در بررسی امضای خود شکست می‌خورد، و رسید بعدی در بررسی پیوند زنجیره‌ای خود شکست می‌خورد (زیرا `previous_receipt_hash` آن دیگر با هش رسید وسط دستکاری شده مطابقت ندارد).


In [13]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): INVALID (failed: signature)
Receipt 2 (   confirm_booking): INVALID (failed: chain link)


رسید 0 هنوز تایید می‌شود (تغییری نکرده و پیش‌نیازی برای وابستگی ندارد). رسید 1 بررسی امضایش را رد می‌کند چون ما `tool_args_hash` را تغییر دادیم. رسید 2 بررسی پیوند زنجیره‌ای خود را رد می‌کند چون `previous_receipt_hash` آن بر اساس رسید (اصلی که حالا تغییر کرده) 1 محاسبه شده است.

حتی اگر مهاجم دوباره رسید 1 تغییر یافته را امضا کند، امضای جدید در برابر کلید صادرکننده رد می‌شود مگر اینکه کلید خصوصی صادرکننده را داشته باشد (یک امضای مجدد با کلید خودش رد می‌شود، همانطور که در بخش 2 نشان داده شد)، و عدم تطابق پیوند زنجیره‌ای در رسید 2 هنوز تغییر را آشکار می‌کند. برای مخفی کردن تغییر، مهاجم باید از نقطه تغییر به بعد هر رسید را دوباره امضا کند که نیازمند داشتن کلید خصوصی صادرکننده است.


## بخش ۴: پیچیدن فراخوانی ابزار عامل با امضای رسید

در یک استقرار واقعی، نمی‌خواهید هر نویسنده عاملی به یاد داشته باشد که `make_receipt` را فراخوانی کند. شما می‌خواهید امضای رسید به‌طور خودکار برای هر فراخوانی ابزار انجام شود.

در اینجا ساده‌ترین الگو آمده است: یک کلاس پوسته که هر تابع ابزاری قابل فراخوانی را می‌گیرد و نسخه‌ای که رسید صادر می‌کند را برمی‌گرداند. این با هر چارچوب عاملی، از جمله چارچوب عامل مایکروسافت (`agent_framework.foundry`) سازگار است.

اگر پروژه Microsoft Foundry راه‌اندازی نکرده‌اید، نمونه محلی شبیه‌سازی شده زیر همچنان الگو را نشان می‌دهد.


In [14]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [15]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


Tool was called 3 times.
Each call produced a signed receipt linked to the previous one.

Receipt 0 (lookup_flights): VALID
Receipt 1 (lookup_flights): VALID
Receipt 2 (lookup_flights): VALID


### ادغام با چارچوب عامل مایکروسافت

لفاف `ReceiptedTool` بالا مستقل از چارچوب است. برای استفاده از آن درون یک عاملی که با چارچوب عامل مایکروسافت ساخته شده است، تابع لفاف شده را به عنوان یک ابزار ثبت کنید. یک طرح کلی (شما جایگزین ماک با یک ثبت ابزار واقعی Microsoft Foundry می‌کنید):

```python
# Pseudocode showing the integration shape.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="You are a Contoso Travel agent ...",
#     tools=[receipted_lookup],   # the wrapped tool, not the raw function
# )
# response = agent.run("Find flights from Sydney to Los Angeles in June.")
#
# # After the run, every tool call the agent made has a signed receipt:
# audit_chain = receipted_lookup.receipts
```

چارچوب عامل نیازی به دانستن چیزی درباره رسیدها ندارد. امضای رسید دور ابزار پیچیده می‌شود، نه اینکه به چارچوب متصل شود. این نحوه افزودن منشاء به کد عامل موجود بدون بازنویسی عامل است.


## Recap and stretch challenge

You have:

- Generated an Ed25519 key pair.
- Built and signed a receipt for an agent tool call.
- Verified the receipt offline using only the issuer's public key, obtained independently of the receipt.
- Tampered with a receipt and observed verification fail.
- Built a hash-chained sequence of three receipts.
- Tampered with the middle of the chain and observed both signature failure and chain-link failure.
- Wrapped a tool function with automatic receipt signing.

**Stretch challenge.** Extend the receipt schema with a `request_id` field (a UUID for distributed tracing). Update `make_receipt` to include it, and confirm receipts still verify end to end. Then modify the field after signing and confirm verification fails. This forces you to internalize how every byte of the canonical encoding contributes to the signature.

**Important boundary.** Under a pinned key and chain context, receipts prove signature attribution, content integrity, and a link to the presented predecessor. They do NOT prove execution, source truth, trusted time, completeness, policy evaluation, or correctness. Receipts are a foundation. Governance is the system you build on top.

Read the lesson README again with that boundary in mind. The most common mistake teams make with receipts is assuming "we have receipts" means "we are governed." It does not. Receipts make agent behavior auditable. They do not make it correct.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوء تفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
